<a href="https://colab.research.google.com/github/aminefrikha38/MachineLearningProject/blob/main/Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.datasets import cifar10
from PIL import Image
import requests, io

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = cifar10.load_data()

y_train = y_train_raw.flatten()
y_test  = y_test_raw.flatten()

CLASS_NAMES = ['airplane','car','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print(f"Train: {X_train_raw.shape}  Test: {X_test_raw.shape}")

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, idx in zip(axes.flat, np.random.choice(50000, 10, replace=False)):
    ax.imshow(X_train_raw[idx])
    ax.set_title(CLASS_NAMES[y_train[idx]], fontsize=9)
    ax.axis('off')
plt.suptitle("10 random CIFAR-10 images", fontsize=12)
plt.tight_layout()
plt.show()


def softmax(O):
    shifted = O - O.max(axis=1, keepdims=True)
    exp_O   = np.exp(shifted)
    return exp_O / exp_O.sum(axis=1, keepdims=True)

def one_hot(y, n_classes=10):
    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1.0
    return Y

def cross_entropy_loss(P, Y_oh):
    return -np.sum(Y_oh * np.log(np.clip(P, 1e-12, 1.0))) / P.shape[0]

def error_rate(P, y):
    return np.mean(np.argmax(P, axis=1) != y)

def relu(x):      return np.maximum(0, x)
def relu_grad(x): return (x > 0).astype(float)


class LinearModel:
    def __init__(self, n_features, n_classes=10):
        rng = np.random.default_rng(42)
        self.A = rng.normal(0, 0.01, (n_classes, n_features))
        self.b = np.zeros(n_classes)

    def forward(self, X):
        return softmax(X @ self.A.T + self.b)

    def train(self, X, y, lr=0.1, n_epochs=20, batch_size=256, verbose=True):
        Y_oh = one_hot(y); n = len(X); losses = []
        for epoch in range(n_epochs):
            idx = np.random.permutation(n)
            for start in range(0, n, batch_size):
                Xb = X[idx[start:start+batch_size]]
                Yb = Y_oh[idx[start:start+batch_size]]
                nb = len(Xb)
                P  = self.forward(Xb)
                delta = (P - Yb) / nb
                self.A -= lr * (delta.T @ Xb)
                self.b -= lr * delta.sum(axis=0)
            P_full = self.forward(X)
            loss = cross_entropy_loss(P_full, Y_oh)
            losses.append(loss)
            if verbose and (epoch % 5 == 0 or epoch == n_epochs-1):
                print(f"  Epoch {epoch+1:3d}/{n_epochs}  loss={loss:.4f}  "
                      f"train_err={error_rate(P_full, y):.4f}")
        return losses


class MLPModel:
    def __init__(self, n_features, hidden_sizes=[128], n_classes=10):
        rng = np.random.default_rng(42)
        self.layers = []
        sizes = [n_features] + hidden_sizes + [n_classes]
        for i in range(len(sizes)-1):
            std = np.sqrt(2.0 / sizes[i])
            W = rng.normal(0, std, (sizes[i+1], sizes[i]))
            b = np.zeros(sizes[i+1])
            self.layers.append([W, b])

    def forward(self, X):
        cache = {'A0': X}; A = X
        for i, (W, b) in enumerate(self.layers[:-1]):
            O = A @ W.T + b; A = relu(O)
            cache[f'O{i+1}'] = O; cache[f'A{i+1}'] = A
        W, b = self.layers[-1]
        P = softmax(A @ W.T + b)
        cache[f'A{len(self.layers)}'] = P
        return P, cache

    def backward(self, cache, Y_oh):
        n = Y_oh.shape[0]; n_l = len(self.layers); grads = [None]*n_l
        delta = (cache[f'A{n_l}'] - Y_oh) / n
        for i in reversed(range(n_l)):
            A_prev = cache[f'A{i}']; W, _ = self.layers[i]
            grads[i] = (delta.T @ A_prev, delta.sum(axis=0))
            if i > 0:
                delta = (delta @ W) * relu_grad(cache[f'O{i}'])
        return grads

    def train(self, X, y, lr=0.05, n_epochs=20, batch_size=256, verbose=True):
        Y_oh = one_hot(y); n = len(X); losses = []
        for epoch in range(n_epochs):
            idx = np.random.permutation(n)
            for start in range(0, n, batch_size):
                Xb = X[idx[start:start+batch_size]]
                Yb = Y_oh[idx[start:start+batch_size]]
                P, cache = self.forward(Xb)
                for i, (dW, db) in enumerate(self.backward(cache, Yb)):
                    self.layers[i][0] -= lr * dW
                    self.layers[i][1] -= lr * db
            P_full, _ = self.forward(X)
            loss = cross_entropy_loss(P_full, Y_oh)
            losses.append(loss)
            if verbose and (epoch % 5 == 0 or epoch == n_epochs-1):
                print(f"  Epoch {epoch+1:3d}/{n_epochs}  loss={loss:.4f}  "
                      f"train_err={error_rate(P_full, y):.4f}")
        return losses


def to_grayscale(X_rgb):
    R = X_rgb[:,:,:,0].astype(float)
    G = X_rgb[:,:,:,1].astype(float)
    B = X_rgb[:,:,:,2].astype(float)
    gray = 0.299*R + 0.587*G + 0.114*B
    return gray.reshape(len(X_rgb), -1) / 255.0

X_train_gray = to_grayscale(X_train_raw)
X_test_gray  = to_grayscale(X_test_raw)
print(f"Grayscale train: {X_train_gray.shape}")

fig, axes = plt.subplots(2, 5, figsize=(12,5))
idxs = np.random.choice(50000, 5, replace=False)
for i, idx in enumerate(idxs):
    axes[0,i].imshow(X_train_raw[idx])
    axes[0,i].set_title(CLASS_NAMES[y_train[idx]], fontsize=8)
    axes[0,i].axis('off')
    axes[1,i].imshow(X_train_gray[idx].reshape(32,32), cmap='gray')
    axes[1,i].axis('off')
axes[0,0].set_ylabel('Color', fontsize=9)
axes[1,0].set_ylabel('Grayscale', fontsize=9)
plt.suptitle("Color vs Grayscale conversion", fontsize=11)
plt.tight_layout(); plt.show()

lin_gray = LinearModel(n_features=1024)
lin_gray.train(X_train_gray, y_train, lr=0.1, n_epochs=20)
P_test_lin_gray = lin_gray.forward(X_test_gray)
err_lin_gray = error_rate(P_test_lin_gray, y_test)
print(f"\n→ Linear (grayscale) test error: {err_lin_gray:.4f}  accuracy: {(1-err_lin_gray)*100:.1f}%")

mlp_gray = MLPModel(n_features=1024, hidden_sizes=[256])
mlp_gray.train(X_train_gray, y_train, lr=0.05, n_epochs=20)
P_test_mlp_gray, _ = mlp_gray.forward(X_test_gray)
err_mlp_gray = error_rate(P_test_mlp_gray, y_test)
print(f"\n→ Neural network 1L (grayscale) test error: {err_mlp_gray:.4f}  accuracy: {(1-err_mlp_gray)*100:.1f}%")


X_train_rgb = X_train_raw.reshape(50000, -1).astype(float) / 255.0
X_test_rgb  = X_test_raw.reshape(10000, -1).astype(float)  / 255.0
print(f"RGB train: {X_train_rgb.shape}")

lin_rgb = LinearModel(n_features=3072)
lin_rgb.train(X_train_rgb, y_train, lr=0.05, n_epochs=20)
P_test_lin_rgb = lin_rgb.forward(X_test_rgb)
err_lin_rgb = error_rate(P_test_lin_rgb, y_test)
print(f"\n→ Linear (RGB) test error: {err_lin_rgb:.4f}  accuracy: {(1-err_lin_rgb)*100:.1f}%")

mlp_rgb = MLPModel(n_features=3072, hidden_sizes=[256])
mlp_rgb.train(X_train_rgb, y_train, lr=0.05, n_epochs=20)
P_test_mlp_rgb, _ = mlp_rgb.forward(X_test_rgb)
err_mlp_rgb = error_rate(P_test_mlp_rgb, y_test)
print(f"\n→ Neural network 1L (RGB) test error: {err_mlp_rgb:.4f}  accuracy: {(1-err_mlp_rgb)*100:.1f}%")


def apply_convolution(image_gray, K, l=0):
    H, W = image_gray.shape
    padded = np.pad(image_gray, pad_width=1, mode='constant', constant_values=0)
    output = np.zeros((H, W))
    for u in range(H):
        for v in range(W):
            patch = padded[u:u+3, v:v+3]
            output[u, v] = np.sum(K * patch) + l
    return np.clip(output, 0, 255)

filters = {
    'K1 — blur (average)':  (1/9) * np.array([[1,1,1],[1,1,1],[1,1,1]]),
    'K2 — sharpen':          np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]),
    'K3 — horizontal edges': np.array([[-1,2,-1],[-1,2,-1],[-1,2,-1]]),
    'K4 — vertical edges':   np.array([[-1,0,1],[-1,0,1],[-1,0,1]]),
    'K5 — Sobel (vertical)': np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
    'K6 — diagonal edges':   np.array([[-2,-1,0],[-1,1,1],[0,1,2]]),
}

demo_img_gray = (X_train_gray[0].reshape(32, 32) * 255).astype(float)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0,0].imshow(demo_img_gray, cmap='gray')
axes[0,0].set_title('Original', fontsize=10, fontweight='bold')
axes[0,0].axis('off')
for ax, (name, K) in zip(list(axes.flat)[1:], filters.items()):
    filtered = apply_convolution(demo_img_gray, K)
    ax.imshow(filtered, cmap='gray')
    ax.set_title(name, fontsize=8)
    ax.axis('off')
axes[1,3].axis('off')
plt.suptitle("Effect of the 6 convolution filters (§2.3.2)", fontsize=12)
plt.tight_layout()
plt.savefig('cifar_convolution_demo.png', dpi=150)
plt.show()


class CIFAR_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3,  out_channels=64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.fc    = nn.Linear(in_features=4096, out_features=10)
        self.relu  = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.relu(self.conv3(x))
        x = self.pool(x)
        x = self.relu(self.conv4(x))
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


def to_torch_format(X_raw, y):
    X = torch.tensor(X_raw, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    y = torch.tensor(y, dtype=torch.long)
    return X, y

X_train_t, y_train_t = to_torch_format(X_train_raw, y_train)
X_test_t,  y_test_t  = to_torch_format(X_test_raw,  y_test)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t,  y_test_t)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

print(f"\nPyTorch tensors — Train: {X_train_t.shape}  Test: {X_test_t.shape}")


def train_cnn(model, train_loader, test_loader, n_epochs=15, lr=0.001):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses, train_errs, test_errs = [], [], []

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        correct_train = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits  = model(X_batch)
            loss    = criterion(logits, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss    += loss.item() * len(X_batch)
            correct_train += (logits.argmax(1) == y_batch).sum().item()

        avg_loss  = epoch_loss / len(train_loader.dataset)
        train_err = 1 - correct_train / len(train_loader.dataset)

        model.eval()
        correct_test = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                logits  = model(X_batch)
                correct_test += (logits.argmax(1) == y_batch).sum().item()

        test_err = 1 - correct_test / len(test_loader.dataset)
        train_losses.append(avg_loss)
        train_errs.append(train_err)
        test_errs.append(test_err)

        print(f"  Epoch {epoch+1:2d}/{n_epochs}  "
              f"loss={avg_loss:.4f}  "
              f"train_err={train_err:.4f}  "
              f"test_err={test_err:.4f}")

    return train_losses, train_errs, test_errs


cnn = CIFAR_CNN()
total_params = sum(p.numel() for p in cnn.parameters())
print(f"Total parameters: {total_params:,}")
print(cnn)

train_losses, train_errs, test_errs = train_cnn(
    cnn, train_loader, test_loader, n_epochs=15, lr=0.001)

final_test_err = test_errs[-1]
print(f"\n→ CNN final test error: {final_test_err:.4f}  accuracy: {(1-final_test_err)*100:.1f}%")


results = {
    "Linear — grayscale (1024)":      err_lin_gray,
    "Neural network — grayscale (1024)": err_mlp_gray,
    "Linear — RGB (3072)":            err_lin_rgb,
    "Neural network — RGB (3072)":    err_mlp_rgb,
    "CNN — RGB (PyTorch)":            final_test_err,
}

print(f"\n{'Model':<45} {'Test Error':>10} {'Accuracy':>10}")
print("-"*67)
for name, err in results.items():
    marker = " ★" if "CNN" in name else ""
    print(f"{name:<45} {err*100:>9.2f}%  {(1-err)*100:>9.2f}%{marker}")

print("\nPublished benchmarks (§2.2):")
benchmarks = [
    ("Conv. Deep Belief Nets (2010)", 21.1),
    ("Maxout Networks (2013)",         9.38),
    ("Fractional Max-Pooling (2014)",  3.47),
    ("Dense Connected CNN (2016)",     3.46),
    ("Vision Transformer (2021)",      0.5),
]
for name, err in benchmarks:
    print(f"  {name:<38} {err:>5.2f}%")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, len(train_losses)+1)
ax1.plot(epochs, train_losses, 'b-o', markersize=4, label='Training loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-entropy loss')
ax1.set_title('CNN training loss'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs, [e*100 for e in train_errs], 'b-o', markersize=4, label='Train error %')
ax2.plot(epochs, [e*100 for e in test_errs],  'r-o', markersize=4, label='Test error %')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Error rate (%)')
ax2.set_title('CNN train vs test error'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('CNN training — CIFAR-10', fontsize=12)
plt.tight_layout()
plt.savefig('cifar_cnn_training.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
names  = list(results.keys())
errs   = [v*100 for v in results.values()]
colors = ['#aab4c8','#7f8fa6','#b8c7a0','#7a9e6a','#e07b54']
bars   = ax.bar(names, errs, color=colors, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Test error rate (%)')
ax.set_title('Error rate comparison — CIFAR-10 (all models)')
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
for bar, v in zip(bars, errs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('cifar_comparison.png', dpi=150)
plt.show()


cnn.eval()
all_preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        logits = cnn(X_batch.to(device))
        all_preds.extend(logits.argmax(1).cpu().numpy())

all_preds = np.array(all_preds)
wrong_idx = np.where(all_preds != y_test)[0]

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
fig.suptitle('Examples of misclassified images (CNN)', fontsize=12)
for ax, idx in zip(axes.flat, wrong_idx[:18]):
    ax.imshow(X_test_raw[idx])
    ax.set_title(f'True:{CLASS_NAMES[y_test[idx]]}\nPred:{CLASS_NAMES[all_preds[idx]]}',
                 fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig('cifar_errors.png', dpi=150)
plt.show()

Using device: cpu


Exception: URL fetch failure on https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz: 503 -- Service Unavailable